### Set up 

In [3]:
# ==============================================================================
# CELLULE 1 : SETUP, PARAMÈTRES GLOBAUX ET CHARGEMENT DES DONNÉES GDELT
# ==============================================================================
import csv
from datetime import datetime
import io
from pathlib import Path
import warnings
from group_lasso import GroupLasso
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import requests
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')

# --- PARAMÈTRES GLOBAUX ---
START_DATE = datetime(2013, 1, 1)
END_DATE = datetime(2026, 8, 1)
MIN_INDEX_DATE = '2015-03-01'

# --- CHARGEMENT GDELT (FRANCE / US / UK) ---
dir_geo = Path('./indicators_geo_monthly')
dir_geo_uk = Path('./indicators_geo_monthly_UK')

# 1. Chargement France / US
if dir_geo.exists():
  files_geo = list(dir_geo.glob('*.parquet'))
  df_geo_main = pd.concat(
      [pd.read_parquet(f) for f in files_geo], ignore_index=True
  )
else:
  raise FileNotFoundError('⚠ Dossier ./indicators_geo_monthly introuvable.')

# 2. Chargement UK
if dir_geo_uk.exists():
  files_uk = list(dir_geo_uk.glob('*.parquet'))
  if len(files_uk) > 0:
    df_geo_uk = pd.concat(
        [pd.read_parquet(f) for f in files_uk], ignore_index=True
    )
    df_geo_uk['region_key'] = 'UK'
  else:
    df_geo_uk = pd.DataFrame()
else:
  df_geo_uk = pd.DataFrame()

# 3. Fusion finale
df_geo = pd.concat([df_geo_main, df_geo_uk], ignore_index=True)

if 'region_key' in df_geo.columns:
  df_geo = (
      df_geo.groupby(['period', 'region_key']).max().reset_index().copy()
  )
  df_geo['period'] = pd.to_datetime(df_geo['period'])
  print(f'✓ Base RÉGIONALE GDELT chargée.')
  print(
      f"✓ Régions trouvées en mémoire : {df_geo['region_key'].unique().tolist()}"
  )
  print(f'✓ Lignes totales : {len(df_geo)}')
else:
  print("⚠ ERREUR CRITIQUE : La colonne 'region_key' est absente.")

✓ Base RÉGIONALE GDELT chargée.
✓ Régions trouvées en mémoire : ['Africa', 'China', 'European Union', 'France', 'India', 'Japan', 'Lebanon', 'Middle East', 'North America', 'Russia', 'South America', 'South East Asia', 'UK', 'US']
✓ Lignes totales : 1918


In [4]:
# ==============================================================================
# CELLULE 2 : LIBRAIRIE DES FONCTIONS ÉCONOMÉTRIQUES ET DE PRÉPARATION
# ==============================================================================


def get_filtered_gdelt_cols(df):
  """ÉTAPE 0 : Exclut les variables mères (colinéarité parfaite)."""
  parents_to_exclude = [
      'att_weight_agriculture',
      'att_weight_commodities',
      'att_weight_energy',
      'att_weight_finance',
      'att_weight_industry',
      'att_weight_real_estate',
      'att_weight_tech',
      'att_weight_transport',
  ]
  return [
      col
      for col in df.columns
      if str(col).startswith('att_weight_') and col not in parents_to_exclude
  ]


def strict_stationarize(df):
  """ÉTAPE 1 : Test ADF -> Différence -> Winsorisation si nécessaire."""
  df_stat = df.copy()
  diff_count = 0

  for col in df_stat.columns:
    valid_data = df_stat[col].dropna()
    if len(valid_data) > 10:
      if adfuller(valid_data, autolag='AIC')[1] >= 0.05:
        df_stat[col] = df_stat[col].diff()
        diff_count += 1
        valid_data_diff = df_stat[col].dropna()
        if (
            len(valid_data_diff) > 10
            and adfuller(valid_data_diff, autolag='AIC')[1] >= 0.05
        ):
          p95 = valid_data_diff.quantile(0.95)
          p05 = valid_data_diff.quantile(0.05)
          df_stat[col] = df_stat[col].clip(lower=p05, upper=p95)
          print(f'  ⚕️ Winsorisation appliquée sur : {col}')

  return df_stat.dropna(), diff_count


def fetch_and_prep_macro_dynamic(
    region_config, start_date, end_date, region=None
):
  """Télécharge la macro (FRED/BCE/BoE) et génère les 12 lags d'inflation."""
  macro_config = region_config['macro_vars']
  fred_tickers = {}
  local_tickers = {}

  for var_name, info in macro_config.items():
    if region == 'France' and var_name == 'money':
      continue
    if (
        region == 'UK'
        and info['ticker'] in ['IUMABEDR', 'LPMBD93', 'XUDLUSS']
    ):
      local_tickers[info['ticker']] = var_name
      continue
    fred_tickers[info['ticker']] = var_name

  # --- A. FRED ---
  df = pd.DataFrame()
  if fred_tickers:
    df = web.DataReader(
        list(fred_tickers.keys()), 'fred', start_date, end_date
    )
    df = df.rename(columns=fred_tickers).resample('MS').mean()

  # --- B. BCE (France) ---
  if region == 'France' and 'money' in macro_config:
    ecb_url = 'https://data-api.ecb.europa.eu/service/data/BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E?format=csvdata'
    try:
      df_ecb = pd.read_csv(ecb_url)
      df_ecb['period'] = pd.to_datetime(df_ecb['TIME_PERIOD'])
      df_ecb = df_ecb.set_index('period').rename(columns={'OBS_VALUE': 'money'})
      df_ecb = df_ecb[['money']].resample('MS').mean()
      df = df.join(df_ecb, how='left') if not df.empty else df_ecb
    except Exception as e:
      print(f'⚠ Erreur BCE: {e}')

  # --- C. BoE (UK) ---
  if region == 'UK' and local_tickers:
    df_local_all = pd.DataFrame()
    for ticker, var_name in local_tickers.items():
      file_path = f'data/BoE/{ticker}.csv'
      if Path(file_path).exists():
        try:
          df_temp = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
          df_temp = pd.read_csv(file_path, encoding='ISO-8859-1')

        date_col = df_temp.columns[0]
        df_temp['period'] = pd.to_datetime(df_temp[date_col], errors='coerce')
        df_temp = df_temp.dropna(subset=['period']).set_index('period')

        val_col = next((c for c in df_temp.columns if ticker in c), None)
        if val_col:
          if df_temp[val_col].dtype == object:
            df_temp[val_col] = (
                df_temp[val_col]
                .astype(str)
                .str.replace(',', '', regex=False)
                .str.strip()
            )
          df_temp[val_col] = pd.to_numeric(df_temp[val_col], errors='coerce')
          series = (
              df_temp[[val_col]]
              .rename(columns={val_col: var_name})
              .resample('MS')
              .mean()
          )
          df_local_all = (
              series
              if df_local_all.empty
              else df_local_all.join(series, how='outer')
          )

    if not df_local_all.empty:
      df = df.join(df_local_all, how='outer') if not df.empty else df_local_all

  # --- TRANSFORMATIONS ---
  final_cols = []
  for var_name, info in macro_config.items():
    if var_name not in df.columns:
      continue
    if info['transform'] == 'pct_change':
      df[var_name] = df[var_name].pct_change() * 100
    elif info['transform'] == 'diff':
      df[var_name] = df[var_name].diff()
    final_cols.append(var_name)

  for lag in range(1, 13):
    lag_name = f'inflation_lag{lag}'
    df[lag_name] = df['inflation'].shift(lag)
    final_cols.append(lag_name)

  df = df[(df.index >= start_date) & (df.index <= end_date)]
  return df[final_cols].dropna()


def auto_fwl_orthogonalization(
    X_macro_base, X_lags_pool, X_gdelt, y
):
  """ÉTAPE 2 : FWL avec sélection automatique des retards d'inflation."""
  lag_sequences = [
      [1],
      [1, 2],
      [1, 2, 3],
      [1, 2, 3, 12],
      [1, 2, 3, 4, 12],
      [1, 2, 3, 4, 5, 6, 12],
      list(range(1, 13)),
  ]

  best_pval = -1
  best_y_tilde, best_X_gdelt_tilde, best_lags = None, None, []

  for lags in lag_sequences:
    lag_cols = [
        f'inflation_lag{l}'
        for l in lags
        if f'inflation_lag{l}' in X_lags_pool.columns
    ]
    X_current = pd.concat([X_macro_base, X_lags_pool[lag_cols]], axis=1)

    ols_y = LinearRegression(fit_intercept=True).fit(X_current, y)
    y_tilde = y - ols_y.predict(X_current)

    ols_X = LinearRegression(fit_intercept=True).fit(X_current, X_gdelt)
    X_gdelt_tilde = pd.DataFrame(
        X_gdelt.values - ols_X.predict(X_current),
        columns=X_gdelt.columns,
        index=X_gdelt.index,
    )

    pval = (
        acorr_ljungbox(y_tilde, lags=[12], return_df=True)['lb_pvalue'].values[0]
    )

    if pval > best_pval:
      best_pval = pval
      best_y_tilde = y_tilde
      best_X_gdelt_tilde = X_gdelt_tilde
      best_lags = lags

    if pval >= 0.05:
      return y_tilde, X_gdelt_tilde, lags, pval

  return best_y_tilde, best_X_gdelt_tilde, best_lags, best_pval


def run_yuan_lin_group_lasso(X, y, groups, sectors_names):
  """ÉTAPE 3 : Yuan & Lin (2006) avec critère Cp."""
  n = len(y)
  unique_groups = np.unique(groups)
  ols = LinearRegression(fit_intercept=False).fit(X, y)
  beta_ols = ols.coef_
  sigma2 = np.mean((y - ols.predict(X)) ** 2)

  lambdas = np.logspace(-3, 1, 50)
  best_cp, best_beta = np.inf, None

  for lbd in lambdas:
    gl = GroupLasso(
        groups=groups,
        l1_reg=0.0,
        group_reg=lbd,
        fit_intercept=False,
        n_iter=5000,
        supress_warning=True,
    )
    gl.fit(X.values, y.values.reshape(-1, 1))
    beta_gl = gl.coef_.flatten()
    preds = gl.predict(X.values).flatten()
    rss = np.sum((y - preds) ** 2)

    df_penalty = 0
    for g in unique_groups:
      idx = np.where(groups == g)[0]
      beta_g = beta_gl[idx]
      if np.linalg.norm(beta_g) > 0:
        norm_ols = np.linalg.norm(beta_ols[idx])
        df_penalty += 1 + (np.linalg.norm(beta_g) / norm_ols) * (len(idx) - 1)

    cp = (rss / sigma2) - n + 2 * df_penalty
    if cp < best_cp:
      best_cp = cp
      best_beta = beta_gl
      best_lbd = lbd

  coef_df = pd.DataFrame(
      {'Variable': X.columns, 'Secteur': sectors_names, 'Coefficient': best_beta}
  )
  return (
      coef_df[coef_df['Coefficient'] != 0].sort_values(
          by=['Secteur', 'Coefficient']
      ),
      best_lbd,
      best_cp,
  )


def run_simon_sparse_group_lasso(
    X, y, groups, sectors_names, alphas=[0.05, 0.25, 0.50, 0.75, 0.95]
):
  """ÉTAPE 4 : Simon et al. (2013) - Sparse Group Lasso avec CV."""
  kf = KFold(n_splits=5, shuffle=True, random_state=42)
  lambdas = np.logspace(-3, 0, 50)
  best_mse, best_lbd, best_alpha = np.inf, None, None

  for alpha in alphas:
    for lbd in lambdas:
      sgl = GroupLasso(
          groups=groups,
          l1_reg=lbd * alpha,
          group_reg=lbd * (1 - alpha),
          fit_intercept=False,
          n_iter=1500,
          supress_warning=True,
      )
      mse_scores = []
      for train_idx, test_idx in kf.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        sgl.fit(X_tr.values, y_tr.values.reshape(-1, 1))
        preds = sgl.predict(X_te.values).flatten()
        mse_scores.append(np.mean((y_te.values - preds) ** 2))

      avg_mse = np.mean(mse_scores)
      if avg_mse < best_mse:
        best_mse, best_lbd, best_alpha = avg_mse, lbd, alpha

  final_sgl = GroupLasso(
      groups=groups,
      l1_reg=best_lbd * best_alpha,
      group_reg=best_lbd * (1 - best_alpha),
      fit_intercept=False,
      n_iter=5000,
      supress_warning=True,
  )
  final_sgl.fit(X.values, y.values.reshape(-1, 1))

  coef_df = pd.DataFrame({
      'Variable': X.columns,
      'Secteur': sectors_names,
      'Coefficient': final_sgl.coef_.flatten(),
  })
  return (
      coef_df[coef_df['Coefficient'] != 0].sort_values(
          by=['Secteur', 'Coefficient']
      ),
      best_lbd,
      best_alpha,
  )


def run_post_lasso_ols(X, y, active_vars_df):
  """ÉTAPE 5 : Inférence Post-Lasso OLS."""
  selected_features = active_vars_df['Variable'].tolist()
  X_selected_with_const = sm.add_constant(X[selected_features])
  results = sm.OLS(y.values, X_selected_with_const.values).fit()

  feature_names = ['Intercept'] + selected_features
  post_lasso_df = pd.DataFrame({
      'Variable': feature_names,
      'Coefficient': results.params,
      'P-value': results.pvalues,
      'CI_Lower': results.conf_int()[:, 0],
      'CI_Upper': results.conf_int()[:, 1],
  })
  return post_lasso_df, results.rsquared, results.rsquared_adj

In [5]:
# ==============================================================================
# CELLULE 3 : CONFIGURATION DES MODÈLES RÉGIONAUX
# ==============================================================================
CONFIG_REGIONS = {
    'France': {
        'macro_vars': {
            'inflation': {'ticker': 'CP0000FRM086NEST', 'transform': 'pct_change'},
            'rate': {'ticker': 'IRSTCI01FRM156N', 'transform': 'diff'},
            'unemp': {'ticker': 'LRHUTTTTFRM156S', 'transform': 'diff'},
            'indpro': {'ticker': 'FRAPRINTO01GYSAM', 'transform': 'pct_change'},
            'money': {'ticker': 'BCE_M3_API', 'transform': 'none'},
            'fx': {'ticker': 'DEXUSEU', 'transform': 'pct_change'},
            'mich': {'ticker': 'CSINFT02FRM460S', 'transform': 'diff'},
            'oil': {'ticker': 'DCOILBRENTEU', 'transform': 'pct_change'},
        },
        'deseasonalize_y': True,
    },
    'US': {
        'macro_vars': {
            'inflation': {'ticker': 'CPIAUCSL', 'transform': 'pct_change'},
            'rate': {'ticker': 'FEDFUNDS', 'transform': 'diff'},
            'unemp': {'ticker': 'UNRATE', 'transform': 'diff'},
            'indpro': {'ticker': 'INDPRO', 'transform': 'pct_change'},
            'money': {'ticker': 'M2SL', 'transform': 'pct_change'},
            'fx': {'ticker': 'DEXUSEU', 'transform': 'pct_change'},
            'mich': {'ticker': 'MICH', 'transform': 'diff'},
            'oil': {'ticker': 'WTISPLC', 'transform': 'pct_change'},
        },
        'deseasonalize_y': False,
    },
    'UK': {
        'macro_vars': {
            'inflation': {'ticker': 'GBRCPIALLMINMEI', 'transform': 'pct_change'},
            'rate': {'ticker': 'IUMABEDR', 'transform': 'diff'},
            'unemp': {'ticker': 'LRHUTTTTGBM156S', 'transform': 'diff'},
            'indpro': {'ticker': 'GBRPRINTO01GYSAM', 'transform': 'pct_change'},
            'money': {'ticker': 'LPMBD93', 'transform': 'pct_change'},
            'fx': {'ticker': 'XUDLUSS', 'transform': 'pct_change'},
            'oil': {'ticker': 'DCOILBRENTEU', 'transform': 'pct_change'},
        },
        'deseasonalize_y': False,
    },
}

In [7]:
# ==============================================================================
# CELLULE 4 : EXÉCUTION DU PIPELINE COMPLET
# ==============================================================================
final_results = {}

for region, config in CONFIG_REGIONS.items():
  print(f'\n' + '=' * 70)
  print(f' PIPELINE ACADÉMIQUE COMPLET : {region.upper()}')
  print(f'======================================================================')

  df_macro = fetch_and_prep_macro_dynamic(
      config, START_DATE, END_DATE, region=region
  )
  df_macro = df_macro[df_macro.index >= MIN_INDEX_DATE]

  cols_gdelt = get_filtered_gdelt_cols(df_geo)
  df_gdelt_raw = (
      df_geo[df_geo['region_key'] == region]
      .set_index('period')[cols_gdelt]
  )

  col_to_drop = 'att_weight_finance_international_orgs'
  if col_to_drop in df_gdelt_raw.columns:
    df_gdelt_raw = df_gdelt_raw.drop(columns=[col_to_drop])

  df_gdelt_stat, diff_count = strict_stationarize(df_gdelt_raw)
  print(
      f'✓ ÉTAPE 0 & 1 : {diff_count} variables stationnarisées. Variables mères'
      ' exclues.'
  )

  df_final = df_gdelt_stat.join(df_macro, how='inner')
  y_raw = df_final['inflation']

  if config['deseasonalize_y']:
    decomp = seasonal_decompose(y_raw, model='additive', period=12)
    y_corr = (y_raw - decomp.seasonal).dropna()
  else:
    y_corr = y_raw.dropna()

  X_full = df_final.drop(columns=['inflation'])
  y_corr, X_contemp = y_corr.align(X_full, join='inner')

  scaler = StandardScaler()
  X_scaled = pd.DataFrame(
      scaler.fit_transform(X_contemp),
      columns=X_contemp.columns,
      index=X_contemp.index,
  )

  macro_base_cols = [
      col
      for col in X_scaled.columns
      if col not in cols_gdelt and not col.startswith('inflation_lag')
  ]
  lag_pool_cols = [col for col in X_scaled.columns if col.startswith('inflation_lag')]

  X_macro_base = X_scaled[macro_base_cols]
  X_lags_pool = X_scaled[lag_pool_cols]
  X_gdelt = X_scaled[[col for col in X_scaled.columns if col in cols_gdelt]]

  y_tilde, X_gdelt_tilde, optimal_lags, pval = auto_fwl_orthogonalization(
      X_macro_base, X_lags_pool, X_gdelt, y_corr
  )
  print(
      f'✓ ÉTAPE 2 (FWL Auto) : Lags retenus = {optimal_lags}. Ljung-Box p-value'
      f' = {pval:.4f}'
  )

  secteurs = [col.split('_')[2] for col in X_gdelt_tilde.columns]
  sector_to_id = {sec: i for i, sec in enumerate(list(set(secteurs)))}
  groups_array = np.array([sector_to_id[sec] for sec in secteurs])

  print(
      '\n[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...'
  )
  active_gl, lbd_gl, cp_gl = run_yuan_lin_group_lasso(
      X_gdelt_tilde, y_tilde, groups_array, secteurs
  )
  print(f'  -> Lambda optimal : {lbd_gl:.4f} (Critère Cp min : {cp_gl:.2f})')
  print(f'  -> {len(active_gl)} variables conservées.')
  
  # --- NOUVEAU : Inférence Post-Lasso OLS pour le Group Lasso ---
  if not active_gl.empty:
      print("\n  [Tribunal Statistique du Group Lasso]")
      post_lasso_gl_df, r2_gl, r2_adj_gl = run_post_lasso_ols(
          X_gdelt_tilde, y_tilde, active_gl
      )
      print(f'  -> R-squared Ajusté (Group Lasso) : {r2_adj_gl:.4f}\n')
      print(post_lasso_gl_df.to_string(index=False))

    
  print('\n[ÉTAPE 4] Sparse Group Lasso (Tuning bidimensionnel via CV)...')
  active_sgl, lbd_sgl, alpha_sgl = run_simon_sparse_group_lasso(
      X_gdelt_tilde, y_tilde, groups_array, secteurs
  )
  print(
      f'  -> Lambda optimal : {lbd_sgl:.4f} | Alpha optimal : {alpha_sgl:.2f}'
  )

  if active_sgl.empty:
    print('  -> Résultat : Aucune variable conservée.')
  else:
    print(f'  -> {len(active_sgl)} variables conservées.')
    print('\n[ÉTAPE 5] Tribunal Statistique : Inférence Post-Lasso OLS...')
    post_lasso_df, r2_sgl, r2_adj_sgl = run_post_lasso_ols(
        X_gdelt_tilde, y_tilde, active_sgl
    )
    print(f'  -> R-squared (Médias) : {r2_sgl:.4f}')
    print(f'  -> R-squared Ajusté   : {r2_adj_sgl:.4f}\n')
    print(post_lasso_df.to_string(index=False))

    final_results[region] = {
        'SGL_Selection': active_sgl,
        'OLS_Inference': post_lasso_df,
        'R2_adj': r2_adj_sgl,
        'Lags_Retenus': optimal_lags,
    }


 PIPELINE ACADÉMIQUE COMPLET : FRANCE
✓ ÉTAPE 0 & 1 : 22 variables stationnarisées. Variables mères exclues.
✓ ÉTAPE 2 (FWL Auto) : Lags retenus = [1, 2, 3, 12]. Ljung-Box p-value = 0.2963

[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...
  -> Lambda optimal : 0.0026 (Critère Cp min : 73.38)
  -> 50 variables conservées.

  [Tribunal Statistique du Group Lasso]
  -> R-squared Ajusté (Group Lasso) : 0.1824

                                  Variable   Coefficient  P-value  CI_Lower  CI_Upper
                                 Intercept  1.951564e-17 1.000000 -0.046206  0.046206
         att_weight_agriculture_regulation -4.756975e-02 0.155012 -0.113560  0.018421
        att_weight_agriculture_rural_labor -5.139811e-02 0.512914 -0.207242  0.104446
       att_weight_agriculture_agribusiness -2.999898e-02 0.763770 -0.228269  0.168271
att_weight_agriculture_infrastructure_tech  2.109698e-03 0.954903 -0.072014  0.076234
              att_weight_agriculture_water  4.602246e